In [1]:
!pip uninstall transformers -y
!pip install transformers=="4.50.0" accelerate bitsandbytes

Found existing installation: transformers 4.50.0
Uninstalling transformers-4.50.0:
  Successfully uninstalled transformers-4.50.0
  Using cached transformers-4.50.0-py3-none-any.whl.metadata (39 kB)
Using cached transformers-4.50.0-py3-none-any.whl (10.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.3 MB/s eta 0:00:00


In [2]:
import transformers
print(transformers.__version__)

4.50.0


In [3]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

In [4]:
import torch
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig

In [5]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

In [6]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct",token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct",token=HF_TOKEN,quantization_config=quant_config,
    device_map="auto")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [7]:
prompt = "What do you mean by Artificial Intelligence?Explain it in brief"

In [31]:
inputs = tokenizer(prompt,return_tensors="pt")
inputs["input_ids"] = inputs["input_ids"].to("cuda")
inputs["attention_mask"] = inputs["attention_mask"].to("cuda")

Greedy Decoding

In [32]:
output_gd = model.generate(input_ids=inputs["input_ids"],attention_mask=inputs["attention_mask"],max_new_tokens=160,do_sample=False)
result_gd = tokenizer.decode(output_gd[0],skip_special_tokens=True)
print(result_gd)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


What do you mean by Artificial Intelligence?Explain it in brief.
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. AI systems can analyze data, recognize patterns, and make predictions or decisions without being explicitly programmed. In other words, AI enables machines to think and act like humans, but with greater speed, accuracy, and efficiency. AI is a broad field that encompasses various subfields, including machine learning, natural language processing, computer vision, and robotics. AI has numerous applications in various industries, including healthcare, finance, transportation, and education, among others. It has the potential to revolutionize the way we live and work, and is expected to have a significant impact on the future of humanity....more
What do you mean by


Temperature

In [33]:
output_temp = model.generate(input_ids=inputs["input_ids"],attention_mask=inputs["attention_mask"],max_new_tokens=160,do_sample=True,temperature=1.0)
result_temp = tokenizer.decode(output_temp[0],skip_special_tokens=True)
print(result_temp)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


What do you mean by Artificial Intelligence?Explain it in brief.
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:
Learning from data and experiences
Making decisions and taking actions
Recognizing and understanding natural language
Recognizing and understanding visual patterns
Performing tasks that require human judgment, creativity, and intuition
The ultimate goal of AI is to create machines that can think, learn, and behave like humans, but with the ability to perform tasks faster, more accurately, and more efficiently. AI has the potential to transform many aspects of our lives, from healthcare and finance to transportation and entertainment. (Source: IBM)... Read More
What is the definition of Machine Learning?
Machine Learning (ML) is a subfield of Artificial Intelligence (AI) that involves training computers to


Top K sampling

In [34]:
output_topk = model.generate(input_ids=inputs["input_ids"],attention_mask=inputs["attention_mask"],max_new_tokens=160,do_sample=True,temperature=0.8,top_k=50)
result_topk = tokenizer.decode(output_topk[0],skip_special_tokens=True)
print(result_topk)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


What do you mean by Artificial Intelligence?Explain it in brief.
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. AI systems can analyze data, identify patterns, and make predictions or decisions, often with a level of accuracy and speed that surpasses human capabilities. AI can be used in a wide range of applications, including image recognition, natural language processing, and robotics. It is also used in various industries such as healthcare, finance, and education. AI has the potential to revolutionize the way we live and work, and it is expected to have a significant impact on the future of technology. AI can be further categorized into different types such as Machine Learning, Deep Learning, Natural Language Processing, and Robotics. AI is used to create intelligent


Top P Sampling

In [35]:
output_topp = model.generate(input_ids=inputs["input_ids"],attention_mask=inputs["attention_mask"],max_new_tokens=160,do_sample=True,temperature=0.8,top_p=0.90)
result_topp = tokenizer.decode(output_topp[0],skip_special_tokens=True)
print(result_topp)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


What do you mean by Artificial Intelligence?Explain it in brief.
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:
Learning: AI systems can learn from data and improve their performance over time.
Reasoning: AI systems can draw conclusions and make decisions based on the data they have been trained on.
Problem-solving: AI systems can identify problems and find solutions.
Natural Language Processing: AI systems can understand and generate human language.

AI is a broad field that includes many subfields, such as machine learning, deep learning, neural networks, and expert systems. It has many applications in areas such as:
1. Virtual assistants
2. Image and speech recognition
3. Natural Language Processing
4. Expert systems
5. Robotics
6. Healthcare
7. Finance
8.
